# Bike Sharing Demand - LightGBM 모델링 (MLflow 로깅)

## 실행 방법
- **로컬**: 이 노트북을 직접 실행
- **run_pm.py**: `python run_pm.py --conf-s3-path s3://...` 로 실행 시 papermill이 아래 파라미터를 주입

In [ ]:
# papermill 파라미터 셀 (태그: parameters)
run_id     = None   # run_pm.py가 주입 (없으면 자동 생성)
output_dir = None   # run_pm.py가 주입 (없으면 ./output 사용)

## 1. 환경 설정

In [ ]:
import os
import json
import yaml
import pickle
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib
import mlflow
import mlflow.lightgbm
import boto3
from pathlib import Path
from datetime import datetime
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')
matplotlib.use('Agg')  # non-interactive backend (SageMaker/papermill 호환)

# ── 경로 설정 ──────────────────────────────────────────────────
BASE_DIR   = Path.cwd()
CONF_DIR   = BASE_DIR / 'conf'
DATA_DIR   = BASE_DIR / 'data'

# run_id / output_dir: papermill 주입 또는 로컬 기본값
if run_id is None:
    from uuid import uuid4
    run_id = datetime.now().strftime('%Y%m%d') + '_lightgbm_baseline_' + uuid4().hex[:8]

if output_dir is None:
    output_dir = str(BASE_DIR / 'output' / run_id)

OUTPUT_DIR = Path(output_dir)

# 출력 서브 디렉토리 생성
for sub in ['artifacts/model', 'artifacts/metrics', 'artifacts/charts', 'metadata']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"📁 BASE_DIR  : {BASE_DIR}")
print(f"📁 OUTPUT_DIR: {OUTPUT_DIR}")
print(f"🏷️  run_id    : {run_id}")

## 2. 설정 파일 로드

In [ ]:
def load_yaml(path: Path) -> dict:
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

env_cfg   = load_yaml(CONF_DIR / 'env.yml')
meta_cfg  = load_yaml(CONF_DIR / 'meta.yml')
model_cfg = load_yaml(CONF_DIR / 'model.yml')

SEED        = model_cfg['data']['seed']
TARGET_COL  = model_cfg['features']['target_col']
LOG_TRANSFORM = model_cfg['data'].get('log_transform', True)
HP          = model_cfg['hyperparameters']

print(f"✅ 설정 로드 완료")
print(f"   - project:    {meta_cfg['project']}")
print(f"   - experiment: {meta_cfg['experiment']}")
print(f"   - target_col: {TARGET_COL}")
print(f"   - log_transform: {LOG_TRANSFORM}")
print(f"   - num_iterations: {HP['num_iterations']}")

## 3. 데이터 로드

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'validation.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f"✅ 데이터 로드 완료")
print(f"   - train:      {train_df.shape}")
print(f"   - validation: {val_df.shape}")
print(f"   - test:       {test_df.shape}")
train_df.head(3)

## 4. Feature Engineering

In [ ]:
def add_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    """datetime 컬럼에서 시간 파생 피처 생성"""
    df = df.copy()
    dt = pd.to_datetime(df['datetime'])
    df['hour']         = dt.dt.hour
    df['day']          = dt.dt.day
    df['month']        = dt.dt.month
    df['year']         = dt.dt.year
    df['weekday']      = dt.dt.weekday          # 0=월 ~ 6=일
    df['is_weekend']   = (df['weekday'] >= 5).astype(int)
    df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)
    return df

# casual / registered 는 test set에 존재하지 않으므로 반드시 제거
DROP_COLS = model_cfg['data'].get('drop_col', [])

train_df = add_datetime_features(train_df).drop(columns=DROP_COLS, errors='ignore')
val_df   = add_datetime_features(val_df).drop(columns=DROP_COLS, errors='ignore')
test_df  = add_datetime_features(test_df)

# 피처 컬럼 정의
FEATURE_COLS = (
    model_cfg['features']['numeric_col'] +
    model_cfg['features']['categorical_col'] +
    ['hour', 'day', 'month', 'year', 'weekday', 'is_weekend', 'is_rush_hour']
)

X_train = train_df[FEATURE_COLS]
X_val   = val_df[FEATURE_COLS]
X_test  = test_df[FEATURE_COLS]

# 타겟: log1p 변환 (log_transform=True → RMSLE를 직접 최적화)
if LOG_TRANSFORM:
    y_train = np.log1p(train_df[TARGET_COL])
    y_val   = np.log1p(val_df[TARGET_COL])
else:
    y_train = train_df[TARGET_COL]
    y_val   = val_df[TARGET_COL]

print(f"✅ Feature Engineering 완료")
print(f"   - drop_col:  {DROP_COLS}")
print(f"   - 피처 수:   {len(FEATURE_COLS)}")
print(f"   - 피처 목록: {FEATURE_COLS}")
print(f"   - log_transform: {LOG_TRANSFORM}  →  LightGBM이 RMSLE를 직접 최적화")

## 5. MLflow 설정

In [ ]:
mlflow_cfg = meta_cfg['mlflow']

# MLflow 자격증명 로드 (AWS Secrets Manager)
def get_mlflow_credentials(secret_name: str, region_name: str) -> dict:
    try:
        client = boto3.client('secretsmanager', region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        return json.loads(response['SecretString'])
    except Exception as e:
        print(f"⚠️  Secrets Manager 로드 실패 (로컬 실행 모드): {e}")
        return {}

creds = get_mlflow_credentials(
    secret_name = mlflow_cfg['secret_name'],
    region_name = mlflow_cfg['region_name']
)

if creds:
    os.environ['MLFLOW_TRACKING_USERNAME'] = creds.get('username', mlflow_cfg.get('username', ''))
    os.environ['MLFLOW_TRACKING_PASSWORD'] = creds.get('password', '')

mlflow.set_tracking_uri(mlflow_cfg['tracking_uri'])
mlflow.set_experiment(mlflow_cfg['experiment_name'])

print(f"✅ MLflow 설정 완료")
print(f"   - tracking_uri:    {mlflow_cfg['tracking_uri']}")
print(f"   - experiment_name: {mlflow_cfg['experiment_name']}")

## 6. LightGBM 학습 + MLflow 로깅

In [ ]:
def rmsle(y_true_orig, y_pred_orig):
    """RMSLE 계산 (원래 스케일) — Kaggle 평가 지표"""
    y_true = np.maximum(y_true_orig, 0)
    y_pred = np.maximum(y_pred_orig, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


with mlflow.start_run(run_name=mlflow_cfg.get('run_name') or run_id) as active_run:
    mlflow_run_id = active_run.info.run_id
    print(f"🚀 MLflow Run 시작: {mlflow_run_id}")

    # ── Tags ────────────────────────────────────────────────────
    mlflow.set_tags({
        'project':    meta_cfg['project'],
        'experiment': meta_cfg['experiment'],
        'model':      meta_cfg['model'],
        'run_id':     run_id,
        **{f'tag_{i}': t for i, t in enumerate(mlflow_cfg.get('tags', []))}
    })

    # ── Params ──────────────────────────────────────────────────
    mlflow.log_params(HP)
    mlflow.log_params({
        'log_transform':  LOG_TRANSFORM,
        'n_features':     len(FEATURE_COLS),
        'train_size':     len(X_train),
        'val_size':       len(X_val),
        'seed':           SEED,
    })

    # ── 학습 ────────────────────────────────────────────────────
    lgb_train = lgb.Dataset(X_train, label=y_train)
    lgb_val   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train)

    params = {k: v for k, v in HP.items() if k != 'num_iterations'}
    params['seed']    = SEED
    params['verbose'] = -1

    callbacks = [
        lgb.early_stopping(stopping_rounds=30, verbose=True),
        lgb.log_evaluation(period=50),
    ]

    model = lgb.train(
        params          = params,
        train_set       = lgb_train,
        num_boost_round = HP['num_iterations'],
        valid_sets      = [lgb_train, lgb_val],
        valid_names     = ['train', 'val'],
        callbacks       = callbacks,
    )

    # ── 예측 & 메트릭 (원래 스케일로 복원) ─────────────────────
    val_pred_log  = model.predict(X_val, num_iteration=model.best_iteration)
    val_pred_orig = np.maximum(np.expm1(val_pred_log) if LOG_TRANSFORM else val_pred_log, 0)
    val_true_orig = val_df[TARGET_COL].values

    rmsle_val = rmsle(val_true_orig, val_pred_orig)           # ← 주요 지표
    rmse_val  = np.sqrt(mean_squared_error(val_true_orig, val_pred_orig))
    r2_val    = r2_score(val_true_orig, val_pred_orig)

    metrics = {
        'val_rmsle':      rmsle_val,       # Kaggle 평가 지표 (주요)
        'val_rmse':       rmse_val,
        'val_r2':         r2_val,
        'best_iteration': model.best_iteration,
    }
    mlflow.log_metrics(metrics)

    print(f"\n📊 검증 메트릭")
    print(f"   ★ val_rmsle (주요): {rmsle_val:.4f}")
    print(f"   - val_rmse:         {rmse_val:.4f}")
    print(f"   - val_r2:           {r2_val:.4f}")
    print(f"   - best_iteration:   {model.best_iteration}")

    # ── 아티팩트 저장 ────────────────────────────────────────────
    # 1) 모델 저장
    model_path = OUTPUT_DIR / 'artifacts/model/model.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    mlflow.log_artifact(str(model_path), artifact_path='model')

    # 2) 메트릭 JSON
    metrics_path = OUTPUT_DIR / 'artifacts/metrics/metrics.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    mlflow.log_artifact(str(metrics_path), artifact_path='metrics')

    # 3) Feature Importance 차트
    fi = pd.DataFrame({
        'feature':    model.feature_name(),
        'importance': model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(fi['feature'][:15][::-1], fi['importance'][:15][::-1])
    ax.set_title('Feature Importance (Top 15, Gain)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    fi_chart_path = OUTPUT_DIR / 'artifacts/charts/feature_importance.png'
    fig.savefig(fi_chart_path, dpi=100)
    plt.close(fig)
    mlflow.log_artifact(str(fi_chart_path), artifact_path='charts')

    # 4) Predictions vs Actual 차트 (RMSLE 표기)
    fig2, ax2 = plt.subplots(figsize=(6, 6))
    ax2.scatter(val_true_orig[:500], val_pred_orig[:500], alpha=0.4, s=10)
    max_val = max(val_true_orig.max(), val_pred_orig.max())
    ax2.plot([0, max_val], [0, max_val], 'r--', lw=1)
    ax2.set_xlabel('Actual Count')
    ax2.set_ylabel('Predicted Count')
    ax2.set_title(f'Predictions vs Actual  |  RMSLE={rmsle_val:.4f}')
    plt.tight_layout()
    pred_chart_path = OUTPUT_DIR / 'artifacts/charts/predictions_vs_actual.png'
    fig2.savefig(pred_chart_path, dpi=100)
    plt.close(fig2)
    mlflow.log_artifact(str(pred_chart_path), artifact_path='charts')

    # 5) run_manifest.yml
    manifest = {
        'run_id':        run_id,
        'mlflow_run_id': mlflow_run_id,
        'created_at':    datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        'project':       meta_cfg['project'],
        'experiment':    meta_cfg['experiment'],
        'algorithm':     model_cfg['algorithm']['name'],
        'primary_metric': 'val_rmsle',
        'metrics':       metrics,
    }
    manifest_path = OUTPUT_DIR / 'metadata/run_manifest.yml'
    with open(manifest_path, 'w') as f:
        yaml.dump(manifest, f, allow_unicode=True, default_flow_style=False)
    mlflow.log_artifact(str(manifest_path), artifact_path='metadata')

    print(f"\n✅ MLflow 로깅 완료")
    print(f"   - mlflow_run_id: {mlflow_run_id}")

## 7. 결과 요약

In [ ]:
print("=" * 60)
print("📋 모델링 완료 요약")
print("=" * 60)
print(f"""
🏷️  Run 정보
    - run_id:        {run_id}
    - mlflow_run_id: {mlflow_run_id}

📊 검증 메트릭
    ★ RMSLE (주요/Kaggle): {rmsle_val:.4f}
    - RMSE:                {rmse_val:.4f}
    - R2:                  {r2_val:.4f}
    - best_iteration:      {model.best_iteration}

📁 출력 경로
    {OUTPUT_DIR}

🔗 MLflow
    {mlflow_cfg['tracking_uri']}/#/experiments
""")
print(f"⏰ 완료: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

## 8. 테스트 예측 (선택)

In [ ]:
test_pred_log  = model.predict(X_test, num_iteration=model.best_iteration)
test_pred_orig = np.maximum(np.expm1(test_pred_log) if LOG_TRANSFORM else test_pred_log, 0)

submission = pd.DataFrame({
    'datetime': test_df['datetime'],
    'count':    test_pred_orig.round().astype(int)
})

submission_path = OUTPUT_DIR / 'artifacts/submission.csv'
submission.to_csv(submission_path, index=False)
print(f"✅ submission.csv 저장: {submission_path}")
submission.head()